In [ ]:
# ============================================================
# DEFINE IGBO DIGRAPH UNITIZATION
# ============================================================
import unicodedata

IGBO_DIGRAPHS = {
    "ch", "gb", "gh", "gw", "kp", "kw", "nw", "ny", "sh",
    "Ch", "Gb", "Gh", "Gw", "Kp", "Kw", "Nw", "Ny", "Sh",
    "CH", "GB", "GH", "GW", "KP", "KW", "NW", "NY", "SH"
}

def normalize_nfc(text: str) -> str:
    return unicodedata.normalize("NFC", str(text))

def unitize_morpheme(morpheme: str) -> list:
    norm_m = normalize_nfc(morpheme)
    units = []
    i = 0
    n = len(norm_m)
    while i < n:
        if i + 1 < n and norm_m[i:i+2] in IGBO_DIGRAPHS:
            units.append(norm_m[i:i+2])
            i += 2
        else:
            units.append(norm_m[i])
            i += 1
    return units


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/Thesis")
EXP_DIR = BASE_DIR / "Structural_MorphBPE_Experiment"

DATA_DIR = EXP_DIR / "data"
MODEL_DIR = EXP_DIR / "models"
RESULT_DIR = EXP_DIR / "results"
LOG_DIR = EXP_DIR / "logs"

for directory in [DATA_DIR, MODEL_DIR, RESULT_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Experiment directory:", EXP_DIR)

Experiment directory: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment


In [ ]:
LEXICON_FILE = DATA_DIR / "validated_igbo_lexicon.txt"

print(LEXICON_FILE)
print(LEXICON_FILE.exists())

/content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/data/validated_igbo_lexicon.txt
True


In [ ]:
import pandas as pd

df = pd.read_csv(
    LEXICON_FILE,
    sep="\t",
    header=None,
    names=["word", "segmentation"],
    dtype=str,
    encoding="utf-8"
)

df.head(20)

,word,segmentation
0,gọọmentị,gọọmentị
1,ndị,ndị
2,onwuemeodo,onwu + eme + odo
3,onye,onye
4,aafọ,aafọ
5,aakụkọ,aakụkọ
6,aba,aba
7,ababeghị,a + ba + be + ghị
8,abacha,abacha
9,abagana,abagana


In [ ]:
print("Number of rows:", len(df))
print("Columns:", df.columns.tolist())
print("\nFirst 20 rows:")
display(df.head(20))

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate words:")
print(df["word"].duplicated().sum())

Number of rows: 20318
Columns: ['word', 'segmentation']

First 20 rows:


,word,segmentation
0,gọọmentị,gọọmentị
1,ndị,ndị
2,onwuemeodo,onwu + eme + odo
3,onye,onye
4,aafọ,aafọ
5,aakụkọ,aakụkọ
6,aba,aba
7,ababeghị,a + ba + be + ghị
8,abacha,abacha
9,abagana,abagana



Missing values:
word            0
segmentation    0
dtype: int64

Duplicate words:
46


In [ ]:
df["word"] = df["word"].str.strip()
df["segmentation"] = df["segmentation"].str.strip()

In [ ]:
print("Total rows:", len(df))
print("Unique words:", df["word"].nunique())
print("Missing words:", df["word"].isna().sum())
print("Missing segmentations:", df["segmentation"].isna().sum())
print("Duplicate words:", df["word"].duplicated().sum())

Total rows: 20318
Unique words: 20272
Missing words: 0
Missing segmentations: 0
Duplicate words: 46


In [ ]:
def split_morphemes(segmentation):
    return [
        morpheme.strip()
        for morpheme in segmentation.split("+")
    ]

In [ ]:
examples = [
    "a + ba + be + ghị",
    "a + ba + gbuo + la",
    "a + ba + kwu + ru",
    "gọọmentị"
]

for x in examples:
    print(x, "→", split_morphemes(x))

a + ba + be + ghị → ['a', 'ba', 'be', 'ghị']
a + ba + gbuo + la → ['a', 'ba', 'gbuo', 'la']
a + ba + kwu + ru → ['a', 'ba', 'kwu', 'ru']
gọọmentị → ['gọọmentị']


In [ ]:
def reconstruct_word(segmentation):
    morphemes = split_morphemes(segmentation)
    return "".join(morphemes)

In [ ]:
df["reconstructed"] = df["segmentation"].apply(reconstruct_word)

df["reconstruction_match"] = (
    df["word"] == df["reconstructed"]
)

print(df["reconstruction_match"].value_counts())

reconstruction_match
True     18463
False     1855
Name: count, dtype: int64


In [ ]:
mismatches = df[~df["reconstruction_match"]]

print("Reconstruction mismatches:", len(mismatches))

display(
    mismatches[
        ["word", "segmentation", "reconstructed"]
    ].head(100)
)

Reconstruction mismatches: 1855


,word,segmentation,reconstructed
39,abia,abịa,abịa
40,abiaghi,a + bịa + ghị,abịaghị
41,abiakari,a + bịa + ka + rị,abịakarị
42,abiaziem,a + bịa + zi + e + m,abịaziem
44,abughi,a + bụ + ghị,abụghị
...,...,...,...
1162,akanu-ibiam,akanu + ibiam,akanuibiam
1169,akapamamịrị,akpa + mamịrị,akpamamịrị
1174,akariala,a + ka + rị + a + la,akarịala
1177,akarịala-nysc,a + ka + rị + a + la + nysc,akarịalanysc


In [ ]:
mismatches["difference_length"] = (
    mismatches["reconstructed"].str.len()
    - mismatches["word"].str.len()
)

display(
    mismatches[
        ["word", "segmentation", "reconstructed", "difference_length"]
    ].head(100)
)

/tmp/ipykernel_3162/2883978174.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mismatches["difference_length"] = (


,word,segmentation,reconstructed,difference_length
39,abia,abịa,abịa,0
40,abiaghi,a + bịa + ghị,abịaghị,0
41,abiakari,a + bịa + ka + rị,abịakarị,0
42,abiaziem,a + bịa + zi + e + m,abịaziem,0
44,abughi,a + bụ + ghị,abụghị,0
...,...,...,...,...
1162,akanu-ibiam,akanu + ibiam,akanuibiam,-1
1169,akapamamịrị,akpa + mamịrị,akpamamịrị,-1
1174,akariala,a + ka + rị + a + la,akarịala,0
1177,akarịala-nysc,a + ka + rị + a + la + nysc,akarịalanysc,-1


In [ ]:
punctuation_pattern = r"[-'’.,;:!?/]"

punctuation_cases = df[
    df["word"].str.contains(
        punctuation_pattern,
        regex=True,
        na=False
    )
]

print("Words containing punctuation:",
      len(punctuation_cases))

display(punctuation_cases.head(100))

Words containing punctuation: 3384


,word,segmentation,reconstructed,reconstruction_match
17,abalị-asaa,abalị + - + asaa,abalị-asaa,True
18,abalị-atọ,abalị + - + atọ,abalị-atọ,True
19,abalị-ise,abalị + - + ise,abalị-ise,True
20,abalị-otu,abalị + - + otu,abalị-otu,True
51,abịa-saụt,abịa + - + saụt,abịa-saụt,True
...,...,...,...,...
2031,ana-echikwa,a + na-e + chi + kwa,ana-echikwa,True
2032,ana-efekọ,a + na-e + fe + kọ,ana-efekọ,True
2033,ana-efesa,a + na-e + fe + sa,ana-efesa,True
2034,ana-efote,a + na-e + fo + te,ana-efote,True


In [ ]:
def segmentation_to_structure(segmentation):
    if pd.isna(segmentation):
        return []
    morphemes = split_morphemes(segmentation)

    return [
        unitize_morpheme(morpheme)
        for morpheme in morphemes
    ]


In [ ]:
test_words = [
    "a + ba + be + ghị",
    "a + ba + gbuo + la",
    "a + ba + kwu + ru",
    "gọọmentị"
]

for segmentation in test_words:
    print(segmentation)
    print(segmentation_to_structure(segmentation))
    print()

a + ba + be + ghị
[['a'], ['b', 'a'], ['b', 'e'], ['g', 'h', 'ị']]

a + ba + gbuo + la
[['a'], ['b', 'a'], ['g', 'b', 'u', 'o'], ['l', 'a']]

a + ba + kwu + ru
[['a'], ['b', 'a'], ['k', 'w', 'u'], ['r', 'u']]

gọọmentị
[['g', 'ọ', 'ọ', 'm', 'e', 'n', 't', 'ị']]



In [ ]:
df["morph_structure"] = df["segmentation"].apply(
    segmentation_to_structure
)

In [ ]:
display(
    df[
        ["word", "segmentation", "morph_structure"]
    ].head(30)
)

,word,segmentation,morph_structure
0,gọọmentị,gọọmentị,"[[g, ọ, ọ, m, e, n, t, ị]]"
1,ndị,ndị,"[[n, d, ị]]"
2,onwuemeodo,onwu + eme + odo,"[[o, n, w, u], [e, m, e], [o, d, o]]"
3,onye,onye,"[[o, n, y, e]]"
4,aafọ,aafọ,"[[a, a, f, ọ]]"
5,aakụkọ,aakụkọ,"[[a, a, k, ụ, k, ọ]]"
6,aba,aba,"[[a, b, a]]"
7,ababeghị,a + ba + be + ghị,"[[a], [b, a], [b, e], [g, h, ị]]"
8,abacha,abacha,"[[a, b, a, c, h, a]]"
9,abagana,abagana,"[[a, b, a, g, a, n, a]]"


In [ ]:
TRAIN_FILE = DATA_DIR / "igbo_train_corpus.txt"

In [ ]:
from collections import Counter

with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    train_text = f.read()

train_tokens = train_text.split()

word_freq = Counter(train_tokens)

print("Training tokens:", len(train_tokens))
print("Unique training words:", len(word_freq))

Training tokens: 833843
Unique training words: 54493


In [ ]:
freq_df = pd.DataFrame(
    word_freq.items(),
    columns=["word", "frequency"]
)

display(freq_df.head())

,word,frequency
0,Sowore,91
1,Revolution:,2
2,Ka,670
3,ekwe,157
4,si,2718


In [ ]:
train_lexicon = freq_df.merge(
    df[
        ["word", "segmentation", "morph_structure"]
    ],
    on="word",
    how="left"
)

In [ ]:
annotated = train_lexicon["segmentation"].notna()

print(
    "Training word types:",
    len(train_lexicon)
)

print(
    "Annotated word types:",
    annotated.sum()
)

print(
    "Unannotated word types:",
    (~annotated).sum()
)

print(
    "Type-level annotation coverage:",
    annotated.mean()
)

Training word types: 54539
Annotated word types: 16854
Unannotated word types: 37685
Type-level annotation coverage: 0.309026568143897


In [ ]:
annotated_token_count = train_lexicon.loc[
    annotated, "frequency"
].sum()

total_token_count = train_lexicon["frequency"].sum()

print(
    "Token-level annotation coverage:",
    annotated_token_count / total_token_count
)

Token-level annotation coverage: 0.6436402024119611


In [ ]:
def create_training_structure(row):
    if pd.isna(row["segmentation"]):
        return [unitize_morpheme(row["word"])]

    return row["morph_structure"]


In [ ]:
train_lexicon["structure"] = train_lexicon.apply(
    create_training_structure,
    axis=1
)

In [ ]:
training_corpus = []

for _, row in train_lexicon.iterrows():

    training_corpus.append({
        "word": row["word"],
        "frequency": int(row["frequency"]),
        "morphemes": tuple(
            tuple(m)
            for m in row["structure"]
        )
    })

In [ ]:
for item in training_corpus[:10]:
    print(item)

In [ ]:
from collections import Counter

def get_pair_statistics(corpus):

    pair_counts = Counter()

    for item in corpus:

        frequency = item["frequency"]

        for morpheme in item["morphemes"]:

            for i in range(len(morpheme) - 1):

                pair = (
                    morpheme[i],
                    morpheme[i + 1]
                )

                pair_counts[pair] += frequency

    return pair_counts

In [ ]:
toy_corpus = [
    {
        "word": "eburula",
        "frequency": 100,
        "morphemes": (
            ("e",),
            ("b", "u", "r", "u"),
            ("l", "a")
        )
    }
]

In [ ]:
toy_pairs = get_pair_statistics(toy_corpus)

for pair, frequency in toy_pairs.items():
    print(pair, frequency)

In [ ]:
assert ("e", "b") not in toy_pairs
assert ("u", "l") not in toy_pairs

print("Morphological boundary constraint: PASSED")

In [ ]:
def apply_merge(corpus, target_pair):

    left, right = target_pair

    new_corpus = []

    for item in corpus:

        new_morphemes = []

        for morpheme in item["morphemes"]:

            new_morpheme = []

            i = 0

            while i < len(morpheme):

                if (
                    i < len(morpheme) - 1
                    and morpheme[i] == left
                    and morpheme[i + 1] == right
                ):

                    new_morpheme.append(
                        left + right
                    )

                    i += 2

                else:

                    new_morpheme.append(
                        morpheme[i]
                    )

                    i += 1

            new_morphemes.append(
                tuple(new_morpheme)
            )

        new_corpus.append({
            "word": item["word"],
            "frequency": item["frequency"],
            "morphemes": tuple(new_morphemes)
        })

    return new_corpus

In [ ]:
def get_initial_vocabulary(corpus):

    vocabulary = set()

    for item in corpus:

        for morpheme in item["morphemes"]:

            vocabulary.update(morpheme)

    return vocabulary

In [ ]:
initial_vocab = get_initial_vocabulary(
    training_corpus
)

print("Initial vocabulary size:", len(initial_vocab))
print(sorted(initial_vocab))

In [ ]:
# Words in the training corpus that were not found
# in the manually validated morphological lexicon

unannotated = train_lexicon[
    train_lexicon["segmentation"].isna()
].copy()

print("Unannotated word types:", len(unannotated))
print(
    "Unannotated token occurrences:",
    unannotated["frequency"].sum()
)

print(
    "Percentage of training tokens unannotated:",
    unannotated["frequency"].sum()
    / train_lexicon["frequency"].sum()
)

In [ ]:
unannotated_sorted = unannotated.sort_values(
    "frequency",
    ascending=False
)

display(
    unannotated_sorted[
        ["word", "frequency"]
    ].head(200)
)

In [ ]:
unannotated_sorted["cumulative_frequency"] = (
    unannotated_sorted["frequency"].cumsum()
)

total_tokens = train_lexicon["frequency"].sum()

unannotated_sorted["cumulative_coverage_gain"] = (
    unannotated_sorted["cumulative_frequency"]
    / total_tokens
)

In [ ]:
display(
    unannotated_sorted[
        [
            "word",
            "frequency",
            "cumulative_coverage_gain"
        ]
    ].head(200)
)

In [ ]:
display(
    unannotated_sorted[
        [
            "word",
            "frequency",
            "cumulative_coverage_gain"
        ]
    ].head(200)
)

In [ ]:
for n in [50, 100, 250, 500, 1000, 2000]:

    if n > len(unannotated_sorted):
        continue

    recovered = (
        unannotated_sorted
        .head(n)["frequency"]
        .sum()
    )

    current_coverage = (
        annotated_token_count + recovered
    ) / total_token_count

    print(
        f"Top {n:,} missing word types → "
        f"{current_coverage:.2%} token coverage"
    )

In [ ]:
annotation_candidates = unannotated_sorted[
    ["word", "frequency"]
].copy()

annotation_candidates["segmentation"] = ""

annotation_candidates.to_csv(
    DATA_DIR / "unannotated_training_words_for_validation.txt",
    sep="\t",
    index=False,
    header=False,
    encoding="utf-8"
)

In [ ]:
ADDITIONAL_FILE = DATA_DIR / "Additionally_Validated_fixed.txt"

print("File:", ADDITIONAL_FILE)
print("Exists:", ADDITIONAL_FILE.exists())

File: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/data/Additionally_Validated_fixed.txt
Exists: True


In [ ]:
additional_df = pd.read_csv(
    ADDITIONAL_FILE,
    sep="\t",
    header=None,
    names=["word", "segmentation"],
    dtype=str,
    encoding="utf-8"
)

additional_df["word"] = additional_df["word"].str.strip()
additional_df["segmentation"] = additional_df["segmentation"].str.strip()

print("Additional validated entries:", len(additional_df))

display(additional_df.head(20))

Additional validated entries: 37681


,word,segmentation
0,n’,n'
1,ya,ya
2,a,a
3,otu,otu
4,o,o
5,','
6,Ọ,ọ
7,Ndị,ndị
8,N’,n'
9,Chineke,chineke


In [ ]:
print("Rows:", len(additional_df))
print("Unique words:", additional_df["word"].nunique())
print("Missing words:", additional_df["word"].isna().sum())
print("Missing segmentations:", additional_df["segmentation"].isna().sum())
print("Duplicate words:", additional_df["word"].duplicated().sum())

Rows: 37681
Unique words: 37677
Missing words: 2
Missing segmentations: 2
Duplicate words: 3


In [ ]:
additional_duplicates = additional_df[
    additional_df["word"].duplicated(keep=False)
].sort_values("word")

display(additional_duplicates)

,word,segmentation
997,ígwè,igwe
4109,ígwè,igwe
5214,"ígwè,","igwe ,"
7371,"ígwè,","igwe ,"
374,NaN,na
9583,NaN,NaN


In [ ]:
original_words = set(df["word"].dropna())

additional_df["already_validated"] = (
    additional_df["word"].isin(original_words)
)

print(
    "Already present:",
    additional_df["already_validated"].sum()
)

print(
    "New validated words:",
    (~additional_df["already_validated"]).sum()
)

Already present: 1
New validated words: 37680


In [ ]:
display(
    additional_df[
        additional_df["already_validated"]
    ].head(50)
)

,word,segmentation,already_validated
16508,gbabara,gba + ba + ra,True


In [ ]:
new_additional = additional_df[
    ~additional_df["already_validated"]
].copy()

print("New entries:", len(new_additional))

display(new_additional.head(50))

New entries: 37680


,word,segmentation,already_validated
0,n’,n',False
1,ya,ya,False
2,a,a,False
3,otu,otu,False
4,o,o,False
5,',',False
6,Ọ,ọ,False
7,Ndị,ndị,False
8,N’,n',False
9,Chineke,chineke,False


In [ ]:
combined_lexicon = pd.concat(
    [
        df[["word", "segmentation"]],
        new_additional[["word", "segmentation"]]
    ],
    ignore_index=True
)

In [ ]:
print("Original validated entries:", len(df))
print("New validated entries:", len(new_additional))
print("Combined validated entries:", len(combined_lexicon))
print(
    "Combined unique words:",
    combined_lexicon["word"].nunique()
)

Original validated entries: 20318
New validated entries: 37680
Combined validated entries: 57998
Combined unique words: 57948


In [ ]:
combined_lookup = dict(
    zip(
        combined_lexicon["word"],
        combined_lexicon["segmentation"]
    )
)

In [ ]:
train_lexicon["combined_segmentation"] = (
    train_lexicon["word"].map(combined_lookup)
)

In [ ]:
combined_annotated = (
    train_lexicon["combined_segmentation"].notna()
)

combined_annotated_tokens = train_lexicon.loc[
    combined_annotated,
    "frequency"
].sum()

total_training_tokens = train_lexicon["frequency"].sum()

combined_coverage = (
    combined_annotated_tokens
    / total_training_tokens
)

In [ ]:
print(
    f"New token-level annotation coverage: "
    f"{combined_coverage:.4%}"
)

New token-level annotation coverage: 99.9876%


In [ ]:
previous_coverage = (
    annotated_token_count
    / total_token_count
)

coverage_increase = (
    combined_coverage - previous_coverage
)

print(
    f"Previous coverage: "
    f"{previous_coverage:.4%}"
)

print(
    f"New coverage: "
    f"{combined_coverage:.4%}"
)

print(
    f"Coverage increase: "
    f"{coverage_increase:.4%}"
)

print(
    f"Additional percentage points: "
    f"{coverage_increase * 100:.2f} pp"
)

Previous coverage: 64.3640%
New coverage: 99.9876%
Coverage increase: 35.6236%
Additional percentage points: 35.62 pp


In [ ]:
previous_lookup = dict(
    zip(
        df["word"],
        df["segmentation"]
    )
)

newly_covered = train_lexicon[
    train_lexicon["word"].isin(
        set(new_additional["word"])
    )
].copy()

In [ ]:
newly_recovered_tokens = newly_covered[
    "frequency"
].sum()

print(
    "Newly recovered token occurrences:",
    newly_recovered_tokens
)

Newly recovered token occurrences: 310456


In [ ]:
print(
    "Share of total training tokens recovered:",
    f"{newly_recovered_tokens / total_training_tokens:.4%}"
)

Share of total training tokens recovered: 35.6236%


In [ ]:
newly_covered_ranked = newly_covered.sort_values(
    "frequency",
    ascending=False
)

display(
    newly_covered_ranked[
        [
            "word",
            "frequency",
            "combined_segmentation"
        ]
    ].head(200)
)

,word,frequency,combined_segmentation
34,n’,28453,n'
10,ya,14771,ya
21,a,12127,a
140,otu,5213,otu
114,o,4610,o
...,...,...,...
370,Ụlọikpe,157,ụlọ + ikpe
4886,Gee,156,ge + e
7862,"ụwa,",156,"ụwa ,"
721,ya?,156,ya ?


In [ ]:
remaining_coverage = 1 - combined_coverage

unannotated_after = train_lexicon[
    train_lexicon["combined_segmentation"].isna()
].copy()

remaining_tokens = unannotated_after[
    "frequency"
].sum()

print(
    f"Final annotated coverage: "
    f"{combined_coverage:.4%}"
)

print(
    f"Remaining unannotated coverage: "
    f"{remaining_coverage:.4%}"
)

print(
    "Remaining unannotated token occurrences:",
    remaining_tokens
)

print(
    "Remaining unannotated word types:",
    len(unannotated_after)
)

Final annotated coverage: 99.9876%
Remaining unannotated coverage: 0.0124%
Remaining unannotated token occurrences: 108
Remaining unannotated word types: 13


In [ ]:
remaining_ranked = unannotated_after.sort_values(
    "frequency",
    ascending=False
)

display(
    remaining_ranked[
        ["word", "frequency"]
    ].head(200)
)

,word,frequency
21459,NA,79
44429,ígwé,6
9842,ń,5
43510,"ígwé,",3
15096,Premiere,2
45765,24;,2
25045,4000,2
35288,nan,2
35706,Alan,2
36718,(history),2


In [ ]:
for n in [50, 100, 250, 500, 1000]:

    if n <= len(remaining_ranked):

        additional_tokens = (
            remaining_ranked
            .head(n)["frequency"]
            .sum()
        )

        potential_coverage = (
            combined_annotated_tokens
            + additional_tokens
        ) / total_training_tokens

        print(
            f"Next {n:,} word types → "
            f"potential coverage: "
            f"{potential_coverage:.4%}"
        )

In [ ]:
COMBINED_LEXICON_FILE = (
    DATA_DIR / "Fully_Validated_IGBO_Lexicon.txt"
)

combined_lexicon.to_csv(
    COMBINED_LEXICON_FILE,
    sep="\t",
    header=False,
    index=False,
    encoding="utf-8"
)

print(
    "Saved:",
    COMBINED_LEXICON_FILE
)

Saved: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/data/Fully_Validated_IGBO_Lexicon.txt


In [ ]:
def segmentation_to_structure(segmentation):
    if pd.isna(segmentation):
        return []
    morphemes = split_morphemes(segmentation)

    return [
        unitize_morpheme(morpheme)
        for morpheme in morphemes
    ]


In [ ]:
combined_lexicon["morph_structure"] = (
    combined_lexicon["segmentation"]
    .apply(segmentation_to_structure)
)

In [ ]:
display(
    combined_lexicon[
        ["word", "segmentation", "morph_structure"]
    ].head()
)

,word,segmentation,morph_structure
0,gọọmentị,gọọmentị,"[[g, ọ, ọ, m, e, n, t, ị]]"
1,ndị,ndị,"[[n, d, ị]]"
2,onwuemeodo,onwu + eme + odo,"[[o, n, w, u], [e, m, e], [o, d, o]]"
3,onye,onye,"[[o, n, y, e]]"
4,aafọ,aafọ,"[[a, a, f, ọ]]"


In [ ]:
FINAL_LEXICON_FILE = (
    DATA_DIR / "Fully_Validated_IGBO_Lexicon.txt"
)

combined_lexicon.to_csv(
    FINAL_LEXICON_FILE,
    sep="\t",
    header=False,
    index=False,
    encoding="utf-8"
)

print("Final lexicon saved:", FINAL_LEXICON_FILE)
print("Entries:", len(combined_lexicon))

Final lexicon saved: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/data/Fully_Validated_IGBO_Lexicon.txt
Entries: 57998


In [ ]:
print(
    "Final token-level coverage:",
    f"{combined_coverage:.4%}"
)

Final token-level coverage: 99.9876%


In [ ]:
FINAL_LEXICON_FILE = DATA_DIR / "Fully_Validated_IGBO_Lexicon.txt"

final_df = pd.read_csv(
    FINAL_LEXICON_FILE,
    sep="\t",
    header=None,
    names=["word", "segmentation"],
    dtype=str,
    encoding="utf-8"
)

final_df["word"] = final_df["word"].str.strip()
final_df["segmentation"] = final_df["segmentation"].str.strip()

print("Final lexicon entries:", len(final_df))
print("Unique words:", final_df["word"].nunique())
print("Missing words:", final_df["word"].isna().sum())
print("Missing segmentations:", final_df["segmentation"].isna().sum())

Final lexicon entries: 57998
Unique words: 48059
Missing words: 2
Missing segmentations: 2


In [ ]:
duplicate_words = final_df[
    final_df["word"].duplicated(keep=False)
].sort_values("word")

print("Duplicate word entries:", len(duplicate_words))

display(duplicate_words.head(50))

Duplicate word entries: 15408


,word,segmentation
"'""Ọ","' "" ọ ""","[[""'"", ' ', ' ', '""', ' ', 'ọ', ' ', '""']]"
"'""""Ọ","' "" ọ ""","[[""'"", ' ', ' ', '""', ' ', 'ọ', ' ', '""']]"
"',","' ,","[[""'"", ' ', ',']]"
"'',","' ,","[[""'"", ' ', ',']]"
"'.""""Ndị","' . "" ndị ""","[[""'"", ' ', '.', ' ', ' ', '""', ' ', 'n', 'd',..."
"'.""Ndị","' . "" ndị ""","[[""'"", ' ', '.', ' ', ' ', '""', ' ', 'n', 'd',..."
’;,' ;,"[[""'"", ' ', ';']]"
';,' ;,"[[""'"", ' ', ';']]"
’?,' ?,"[[""'"", ' ', '?']]"
'?,' ?,"[[""'"", ' ', '?']]"


In [ ]:
FINAL_LEXICON_BACKUP = (
    DATA_DIR / "Fully_Validated_IGBO_Lexicon_before_dedup.txt"
)

final_df.to_csv(
    FINAL_LEXICON_BACKUP,
    sep="\t",
    header=False,
    index=False,
    encoding="utf-8"
)

print("Backup saved:", FINAL_LEXICON_BACKUP)

Backup saved: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/data/Fully_Validated_IGBO_Lexicon_before_dedup.txt


In [ ]:
def segmentation_to_structure(segmentation):
    if pd.isna(segmentation):
        return []
    morphemes = split_morphemes(segmentation)

    return [
        unitize_morpheme(morpheme)
        for morpheme in morphemes
    ]


In [ ]:
# Find rows where segmentation is not a string
bad_segmentation = final_df[
    ~final_df["segmentation"].apply(lambda x: isinstance(x, str))
]

print("Non-string segmentation values:", len(bad_segmentation))

display(bad_segmentation)

Non-string segmentation values: 2


,word,segmentation
NaN,NaN,NaN
4000 4000,NaN,NaN


In [ ]:
empty_segmentation = final_df[
    final_df["segmentation"].isna()
    | (final_df["segmentation"].astype(str).str.strip() == "")
]

print("Missing/empty segmentations:", len(empty_segmentation))

display(empty_segmentation)

Missing/empty segmentations: 2


,word,segmentation
NaN,NaN,NaN
4000 4000,NaN,NaN


In [ ]:
print(
    final_df["segmentation"]
    .apply(type)
    .value_counts()
)

segmentation
<class 'str'>      57996
<class 'float'>        2
Name: count, dtype: int64


In [ ]:
def segmentation_to_structure(segmentation):
    if pd.isna(segmentation):
        return []
    morphemes = split_morphemes(segmentation)

    return [
        unitize_morpheme(morpheme)
        for morpheme in morphemes
    ]


In [ ]:
final_df["morph_structure"] = (
    final_df["segmentation"]
    .apply(segmentation_to_structure)
)

In [ ]:
problem_rows = final_df[
    final_df["morph_structure"].apply(len) == 0
]

print("Rows with no morphological structure:", len(problem_rows))

display(problem_rows)

Rows with no morphological structure: 2


,word,segmentation,morph_structure
NaN,NaN,NaN,[]
4000 4000,NaN,NaN,[]


In [ ]:
final_df = final_df[
    final_df["word"].notna()
].copy()

In [ ]:
numeric_entries = final_df[
    final_df["word"].astype(str).str.fullmatch(r"\d+")
]

print("Numeric-only lexicon entries:", len(numeric_entries))

display(numeric_entries)

Numeric-only lexicon entries: 416


,word,segmentation,morph_structure
2019,2019,"[['2', '0', '1', '9']]","[[[, [, ', 2, ', ,, , ', 0, ', ,, , ', 1, ',..."
1,1,[['1']],"[[[, [, ', 1, ', ], ]]]"
2,2,[['2']],"[[[, [, ', 2, ', ], ]]]"
000,000,"[['0', '0', '0']]","[[[, [, ', 0, ', ,, , ', 0, ', ,, , ', 0, ',..."
2018,2018,"[['2', '0', '1', '8']]","[[[, [, ', 2, ', ,, , ', 0, ', ,, , ', 1, ',..."
...,...,...,...
1943,1943,"[['1', '9', '4', '3']]","[[[, [, ', 1, ', ,, , ', 9, ', ,, , ', 4, ',..."
332,332,"[['3', '3', '2']]","[[[, [, ', 3, ', ,, , ', 3, ', ,, , ', 2, ',..."
1587,1587,"[['1', '5', '8', '7']]","[[[, [, ', 1, ', ,, , ', 5, ', ,, , ', 8, ',..."
1553,1553,"[['1', '5', '5', '3']]","[[[, [, ', 1, ', ,, , ', 5, ', ,, , ', 5, ',..."


In [ ]:
# Remove the previously generated structural column
if "morph_structure" in final_df.columns:
    final_df = final_df.drop(
        columns=["morph_structure"]
    )

In [ ]:
def segmentation_to_structure(segmentation):
    if pd.isna(segmentation):
        return []
    morphemes = split_morphemes(segmentation)

    return [
        unitize_morpheme(morpheme)
        for morpheme in morphemes
    ]


In [ ]:
final_df["morph_structure"] = (
    final_df["segmentation"]
    .apply(segmentation_to_structure)
)

In [ ]:
numeric_entries = final_df[
    final_df["word"].astype(str).str.fullmatch(r"\d+")
]

print(
    "Numeric-only entries:",
    len(numeric_entries)
)

display(
    numeric_entries[
        ["word", "segmentation", "morph_structure"]
    ].head(30)
)

Numeric-only entries: 416


,word,segmentation,morph_structure
2019,2019,"[['2', '0', '1', '9']]","[[[, [, ', 2, ', ,, , ', 0, ', ,, , ', 1, ',..."
1,1,[['1']],"[[[, [, ', 1, ', ], ]]]"
2,2,[['2']],"[[[, [, ', 2, ', ], ]]]"
000,000,"[['0', '0', '0']]","[[[, [, ', 0, ', ,, , ', 0, ', ,, , ', 0, ',..."
2018,2018,"[['2', '0', '1', '8']]","[[[, [, ', 2, ', ,, , ', 0, ', ,, , ', 1, ',..."
2017,2017,"[['2', '0', '1', '7']]","[[[, [, ', 2, ', ,, , ', 0, ', ,, , ', 1, ',..."
5,5,[['5']],"[[[, [, ', 5, ', ], ]]]"
2015,2015,"[['2', '0', '1', '5']]","[[[, [, ', 2, ', ,, , ', 0, ', ,, , ', 1, ',..."
20,20,"[['2', '0']]","[[[, [, ', 2, ', ,, , ', 0, ', ], ]]]"
10,10,"[['1', '0']]","[[[, [, ', 1, ', ,, , ', 0, ', ], ]]]"


In [ ]:
# Check structure types
print(
    final_df["morph_structure"]
    .apply(type)
    .value_counts()
)

morph_structure
<class 'list'>    57996
Name: count, dtype: int64


In [ ]:
# Morphologically segmented
display(
    final_df[
        final_df["segmentation"].str.contains(
            "+",
            regex=False,
            na=False
        )
    ][
        ["word", "segmentation", "morph_structure"]
    ].head(20)
)

,word,segmentation,morph_structure


In [ ]:
display(
    final_df[
        final_df["word"].astype(str).str.fullmatch(r"\d+")
    ][
        ["word", "segmentation", "morph_structure"]
    ].head(5000)
)

,word,segmentation,morph_structure
2019,2019,"[['2', '0', '1', '9']]","[[[, [, ', 2, ', ,, , ', 0, ', ,, , ', 1, ',..."
1,1,[['1']],"[[[, [, ', 1, ', ], ]]]"
2,2,[['2']],"[[[, [, ', 2, ', ], ]]]"
000,000,"[['0', '0', '0']]","[[[, [, ', 0, ', ,, , ', 0, ', ,, , ', 0, ',..."
2018,2018,"[['2', '0', '1', '8']]","[[[, [, ', 2, ', ,, , ', 0, ', ,, , ', 1, ',..."
...,...,...,...
1943,1943,"[['1', '9', '4', '3']]","[[[, [, ', 1, ', ,, , ', 9, ', ,, , ', 4, ',..."
332,332,"[['3', '3', '2']]","[[[, [, ', 3, ', ,, , ', 3, ', ,, , ', 2, ',..."
1587,1587,"[['1', '5', '8', '7']]","[[[, [, ', 1, ', ,, , ', 5, ', ,, , ', 8, ',..."
1553,1553,"[['1', '5', '5', '3']]","[[[, [, ', 1, ', ,, , ', 5, ', ,, , ', 5, ',..."


In [ ]:
print(final_df["segmentation"].head(30).to_list())

["[['g', 'ọ', 'ọ', 'm', 'e', 'n', 't', 'ị']]", "[['n', 'd', 'ị']]", "[['o', 'n', 'w', 'u'], ['e', 'm', 'e'], ['o', 'd', 'o']]", "[['o', 'n', 'y', 'e']]", "[['a', 'a', 'f', 'ọ']]", "[['a', 'a', 'k', 'ụ', 'k', 'ọ']]", "[['a', 'b', 'a']]", "[['a'], ['b', 'a'], ['b', 'e'], ['g', 'h', 'ị']]", "[['a', 'b', 'a', 'c', 'h', 'a']]", "[['a', 'b', 'a', 'g', 'a', 'n', 'a']]", "[['a'], ['b', 'a'], ['g', 'b', 'u', 'o'], ['l', 'a']]", "[['a'], ['b', 'a'], ['g', 'h', 'ị']]", "[['a', 'b', 'a', 'h']]", "[['a', 'b', 'a', 'k', 'a', 'l', 'i', 'k', 'i']]", "[['a', 'b', 'a', 'k', 'p', 'a']]", "[['a'], ['b', 'a'], ['k', 'w', 'u'], ['r', 'u']]", "[['a', 'b', 'a', 'l', 'ị']]", "[['a', 'b', 'a', 'l', 'ị'], ['-'], ['a', 's', 'a', 'a']]", "[['a', 'b', 'a', 'l', 'ị'], ['-'], ['a', 't', 'ọ']]", "[['a', 'b', 'a', 'l', 'ị'], ['-'], ['i', 's', 'e']]", "[['a', 'b', 'a', 'l', 'ị'], ['-'], ['o', 't', 'u']]", "[['a', 'b', 'a', 'l', 'ị'], ['a']]", "[['a', 'b', 'a', 'l', 'ị'], ['d', 'ị'], ['e', 'g', 'w', 'u']]", "[['a'], ['b'

In [ ]:
check_df = pd.read_csv(
    FINAL_LEXICON_FILE,
    sep="\t",
    header=None,
    names=["word", "segmentation"],
    dtype=str,
    encoding="utf-8"
)

check_df["word"] = check_df["word"].str.strip()
check_df["segmentation"] = check_df["segmentation"].str.strip()

print(
    "Rows containing + in source:",
    check_df["segmentation"].str.contains(
        "+",
        regex=False,
        na=False
    ).sum()
)

display(
    check_df[
        check_df["segmentation"].str.contains(
            "+",
            regex=False,
            na=False
        )
    ].head(20)
)

Rows containing + in source: 0


,word,segmentation


In [ ]:
import ast

In [ ]:
def parse_morph_structure(value):

    if pd.isna(value):
        return None

    if isinstance(value, list):
        return value

    return ast.literal_eval(value)

In [ ]:
final_df["morph_structure"] = (
    final_df["segmentation"]
    .apply(parse_morph_structure)
)

In [ ]:
display(
    final_df[
        ["word", "segmentation", "morph_structure"]
    ].head(20)
)

,word,segmentation,morph_structure
gọọmentị,gọọmentị,"[['g', 'ọ', 'ọ', 'm', 'e', 'n', 't', 'ị']]","[[g, ọ, ọ, m, e, n, t, ị]]"
ndị,ndị,"[['n', 'd', 'ị']]","[[n, d, ị]]"
onwuemeodo,onwu + eme + odo,"[['o', 'n', 'w', 'u'], ['e', 'm', 'e'], ['o', ...","[[o, n, w, u], [e, m, e], [o, d, o]]"
onye,onye,"[['o', 'n', 'y', 'e']]","[[o, n, y, e]]"
aafọ,aafọ,"[['a', 'a', 'f', 'ọ']]","[[a, a, f, ọ]]"
aakụkọ,aakụkọ,"[['a', 'a', 'k', 'ụ', 'k', 'ọ']]","[[a, a, k, ụ, k, ọ]]"
aba,aba,"[['a', 'b', 'a']]","[[a, b, a]]"
ababeghị,a + ba + be + ghị,"[['a'], ['b', 'a'], ['b', 'e'], ['g', 'h', 'ị']]","[[a], [b, a], [b, e], [g, h, ị]]"
abacha,abacha,"[['a', 'b', 'a', 'c', 'h', 'a']]","[[a, b, a, c, h, a]]"
abagana,abagana,"[['a', 'b', 'a', 'g', 'a', 'n', 'a']]","[[a, b, a, g, a, n, a]]"


In [ ]:
print(
    final_df["morph_structure"]
    .apply(type)
    .value_counts()
)

morph_structure
<class 'list'>    57996
Name: count, dtype: int64


In [ ]:
final_df["num_morphemes"] = (
    final_df["morph_structure"]
    .apply(lambda x: len(x) if isinstance(x, list) else 0)
)

In [ ]:
print(
    final_df["num_morphemes"].value_counts().sort_index()
)

num_morphemes
1     32492
2      5864
3     10241
4      6767
5      2240
6       335
7        42
8         6
9         6
11        3
Name: count, dtype: int64


In [ ]:
print(final_df["morph_structure"].apply(type).value_counts())

print(
    "Multi-unit entries:",
    (final_df["num_morphemes"] > 1).sum()
)

display(
    final_df[
        final_df["num_morphemes"] > 1
    ][["word", "morph_structure"]].head(20)
)

morph_structure
<class 'list'>    57996
Name: count, dtype: int64
Multi-unit entries: 25504


,word,morph_structure
onwuemeodo,onwu + eme + odo,"[[o, n, w, u], [e, m, e], [o, d, o]]"
ababeghị,a + ba + be + ghị,"[[a], [b, a], [b, e], [g, h, ị]]"
abagbuola,a + ba + gbuo + la,"[[a], [b, a], [g, b, u, o], [l, a]]"
abaghị,a + ba + ghị,"[[a], [b, a], [g, h, ị]]"
abakwuru,a + ba + kwu + ru,"[[a], [b, a], [k, w, u], [r, u]]"
abalị-asaa,abalị + - + asaa,"[[a, b, a, l, ị], [-], [a, s, a, a]]"
abalị-atọ,abalị + - + atọ,"[[a, b, a, l, ị], [-], [a, t, ọ]]"
abalị-ise,abalị + - + ise,"[[a, b, a, l, ị], [-], [i, s, e]]"
abalị-otu,abalị + - + otu,"[[a, b, a, l, ị], [-], [o, t, u]]"
abalịa,abalị + a,"[[a, b, a, l, ị], [a]]"


In [ ]:
print(final_df.columns.tolist())
print(final_df.shape)

display(
    final_df[
        ["word", "segmentation", "morph_structure"]
    ].head(20)
)

['word', 'segmentation', 'morph_structure', 'num_morphemes']
(57996, 4)


,word,segmentation,morph_structure
gọọmentị,gọọmentị,"[['g', 'ọ', 'ọ', 'm', 'e', 'n', 't', 'ị']]","[[g, ọ, ọ, m, e, n, t, ị]]"
ndị,ndị,"[['n', 'd', 'ị']]","[[n, d, ị]]"
onwuemeodo,onwu + eme + odo,"[['o', 'n', 'w', 'u'], ['e', 'm', 'e'], ['o', ...","[[o, n, w, u], [e, m, e], [o, d, o]]"
onye,onye,"[['o', 'n', 'y', 'e']]","[[o, n, y, e]]"
aafọ,aafọ,"[['a', 'a', 'f', 'ọ']]","[[a, a, f, ọ]]"
aakụkọ,aakụkọ,"[['a', 'a', 'k', 'ụ', 'k', 'ọ']]","[[a, a, k, ụ, k, ọ]]"
aba,aba,"[['a', 'b', 'a']]","[[a, b, a]]"
ababeghị,a + ba + be + ghị,"[['a'], ['b', 'a'], ['b', 'e'], ['g', 'h', 'ị']]","[[a], [b, a], [b, e], [g, h, ị]]"
abacha,abacha,"[['a', 'b', 'a', 'c', 'h', 'a']]","[[a, b, a, c, h, a]]"
abagana,abagana,"[['a', 'b', 'a', 'g', 'a', 'n', 'a']]","[[a, b, a, g, a, n, a]]"


In [ ]:
from pathlib import Path

TRAINING_CORPUS_FILE = Path(DATA_DIR) / "igbo_train_corpus.txt"

print("Training corpus:", TRAINING_CORPUS_FILE)
print("Exists:", TRAINING_CORPUS_FILE.exists())

Training corpus: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/data/igbo_train_corpus.txt
Exists: True


In [ ]:
with open(TRAINING_CORPUS_FILE, "r", encoding="utf-8") as f:
    corpus_text = f.read()

print("Characters:", len(corpus_text))
print(corpus_text[:1000])

Characters: 4261673
Sowore Revolution: Ka ekwe si akụ maka ngagharịiwe e ji maka ya nwụchie Sowore BBC Igbo kwụ chịm iwetara gị ihe na-aga ka a na-ekwu okwu ngagharịiwe na mpaghara dị iche iche na Naịjirịa.
Ofala Festival: Ka mmemme Ofala sị aga n’ Onitsha taa Oge eruola ọzọ mgbe Obi na-achị Onitsha ga-apụta n’ ebube ya.
Onitsha tanker fire: Ndị ọkụ lara ihe ha n’ iyi na-ekwu etu ọ dị ha Obi nke Onitsha esonyela na ndị na-eti ndị ọkụ tanka a metụtara aka n’ obi.
Hilda Dokubọ kọwara ihe ise dị mkpa nne ọbụla kwesiri ịma Hilda Dokubọ bụ onye na-eme ihe nkiri na-enye ndị nne ezigbo ndụmọdụ.
Presidential Election: A napụrụ ọtụtụ n’ ọwụwa anyanwụ ohere itu vootu-Peter Obi Peter Obi atụọla arịrị na usoro a gbasoro mee ntuliaka n’ Ọwụwa anyanwụ dị iche na nke Ugwu-Awụsa.
Ọgba mbọ: Ihe mere m ji ewe ụmụ nwaanyị n’ ọrụ karịa ụmụnwoke Ụkamaka Okoye kọwara BBC Igbo etu o siri bido n’ iji akpakara igwe rụpụtawa ụzụ igwe nke ọgbatumtum ji arụ ọrụ.
Onye okachamara n’ ihe gbasara nchekwa ' A ga-agbag

In [ ]:
from collections import Counter

corpus_tokens = corpus_text.split()
corpus_freq = Counter(corpus_tokens)

print("Total corpus tokens:", len(corpus_tokens))
print("Unique corpus word types:", len(corpus_freq))

Total corpus tokens: 833843
Unique corpus word types: 54493


In [ ]:
for word, freq in corpus_freq.most_common(30):
    print(f"{word!r}: {freq}")

'na': 39561
'n’': 28453
'ndị': 21238
'bụ': 15602
'ha': 15536
'ya': 14771
'ka': 12534
'a': 12127
'ihe': 11918
'ọ': 11771
'nke': 10000
'ahụ': 7620
'dị': 7041
'ma': 7021
'onye': 6080
'm': 5246
'otu': 5213
'anyị': 4822
'o': 4610
"'": 4605
'mmadụ': 4596
'e': 4386
'ebe': 4365
'maka': 4054
'kwuru': 3851
'Ọ': 3837
'mere': 3722
'aka': 3685
'nwere': 3464
'ji': 3384


In [ ]:
lexicon_words = set(
    final_df["word"]
    .dropna()
    .astype(str)
)

covered_types = set(corpus_freq) & lexicon_words

covered_occurrences = sum(
    corpus_freq[word]
    for word in covered_types
)

total_occurrences = sum(corpus_freq.values())

coverage = covered_occurrences / total_occurrences

print(f"Corpus occurrence coverage: {coverage:.4%}")
print("Covered word types:", len(covered_types))
print("Uncovered word types:", len(corpus_freq) - len(covered_types))
print(
    "Uncovered occurrences:",
    total_occurrences - covered_occurrences
)

Corpus occurrence coverage: 58.9324%
Covered word types: 10962
Uncovered word types: 43531
Uncovered occurrences: 342439


In [ ]:
print("Corpus tokens:", len(corpus_tokens))
print("Corpus types:", len(corpus_freq))

print("\nFirst 50 corpus tokens:")
for token in corpus_tokens[:50]:
    print(repr(token))

Corpus tokens: 833843
Corpus types: 54493

First 50 corpus tokens:
'Sowore'
'Revolution:'
'Ka'
'ekwe'
'si'
'akụ'
'maka'
'ngagharịiwe'
'e'
'ji'
'maka'
'ya'
'nwụchie'
'Sowore'
'BBC'
'Igbo'
'kwụ'
'chịm'
'iwetara'
'gị'
'ihe'
'na-aga'
'ka'
'a'
'na-ekwu'
'okwu'
'ngagharịiwe'
'na'
'mpaghara'
'dị'
'iche'
'iche'
'na'
'Naịjirịa.'
'Ofala'
'Festival:'
'Ka'
'mmemme'
'Ofala'
'sị'
'aga'
'n’'
'Onitsha'
'taa'
'Oge'
'eruola'
'ọzọ'
'mgbe'
'Obi'
'na-achị'


In [ ]:
uncovered = [
    (word, freq)
    for word, freq in corpus_freq.items()
    if word not in lexicon_words
]

uncovered.sort(key=lambda x: x[1], reverse=True)

print("Uncovered types:", len(uncovered))
print("Uncovered occurrences:", sum(freq for _, freq in uncovered))

print("\nMost frequent uncovered tokens:")
for word, freq in uncovered[:100]:
    print(repr(word), freq)

Uncovered types: 43531
Uncovered occurrences: 342439

Most frequent uncovered tokens:
'n’' 28453
'kwuru' 3851
'Ọ' 3837
'mere' 3722
'nwere' 3464
'Ndị' 3140
'N’' 2520
'Chineke' 2410
'Ihe' 2235
'ya.' 2225
'O' 1983
'Naịjirịa' 1804
'bụrụ' 1536
'otú' 1497
'Baịbụl' 1495
'na-eme' 1423
'ahụ.' 1384
'ya,' 1381
'ha.' 1302
'ahụ,' 1281
'Ha' 1245
'nakwa' 1211
'gara' 1145
'gwara' 1127
'A' 1076
'Onye' 1050
'Jehova' 1047
'wee' 1045
'uweojii' 1013
'nyere' 1010
'Akụkọ' 1009
'chọrọ' 994
'gbara' 968
'mee' 953
'Otu' 951
'a.' 927
'Ma,' 925
'mara' 914
'ga-eme' 876
'E' 869
'were' 868
'site' 862
'Mgbe' 851
'Nke' 820
'jiri' 780
'bụ́' 757
'na-ahụ' 716
'hụrụ' 707
'a,' 700
'ga-amasị' 697
'na-ekwu' 688
'ruo' 680
'Ka' 670
'onyeisiala' 668
'Anyị' 635
'siri' 605
'sị:' 603
'banyere' 596
'kwesịrị' 582
'kwuo' 565
'agaghị' 564
'Mana' 557
'dere' 553
'Anambra' 550
'ha,' 548
'ná' 534
'ntuliaka' 533
'ekwuola' 530
'gosiri' 529
'nwee' 524
'ga-esi' 522
'na-agba' 520
'gaa' 519
'anaghị' 514
'ruru' 511
'Na' 506
'eme' 501
'gị:' 497
'n

In [ ]:
covered = [
    (word, freq)
    for word, freq in corpus_freq.items()
    if word in lexicon_words
]

covered.sort(key=lambda x: x[1], reverse=True)

print("\nMost frequent covered tokens:")
for word, freq in covered[:50]:
    print(repr(word), freq)


Most frequent covered tokens:
'na' 39561
'ndị' 21238
'bụ' 15602
'ha' 15536
'ya' 14771
'ka' 12534
'a' 12127
'ihe' 11918
'ọ' 11771
'nke' 10000
'ahụ' 7620
'dị' 7041
'ma' 7021
'onye' 6080
'm' 5246
'otu' 5213
'anyị' 4822
'o' 4610
"'" 4605
'mmadụ' 4596
'e' 4386
'ebe' 4365
'maka' 4054
'aka' 3685
'ji' 3384
'afọ' 3166
'ike' 3142
'ime' 2917
'ọrụ' 2775
'mgbe' 2774
'si' 2718
'ọzọ' 2408
'mba' 2397
'oge' 2334
'ụlọ' 2273
'dịka' 2262
'iri' 2218
'anya' 2203
'steeti' 2177
'niile' 2074
'ụmụ' 2000
'abụọ' 1987
'iche' 1982
'akwụkwọ' 1870
'ị' 1756
'ọtụtụ' 1740
'isi' 1718
'gị' 1715
'obi' 1703
'obodo' 1683


In [ ]:
import string

punctuation_attached = [
    (word, freq)
    for word, freq in uncovered
    if any(char in string.punctuation for char in word)
]

print(
    "Uncovered types containing ASCII punctuation:",
    len(punctuation_attached)
)

print(
    "Their occurrences:",
    sum(freq for _, freq in punctuation_attached)
)

for word, freq in sorted(
    punctuation_attached,
    key=lambda x: x[1],
    reverse=True
)[:50]:
    print(repr(word), freq)

Uncovered types containing ASCII punctuation: 22236
Their occurrences: 107051
'ya.' 2225
'na-eme' 1423
'ahụ.' 1384
'ya,' 1381
'ha.' 1302
'ahụ,' 1281
'a.' 927
'Ma,' 925
'ga-eme' 876
'na-ahụ' 716
'a,' 700
'ga-amasị' 697
'na-ekwu' 688
'sị:' 603
'ha,' 548
'ga-esi' 522
'na-agba' 520
'gị:' 497
'ga-eji' 439
'na-aga' 425
'ọzọ.' 403
'anya.' 397
'na-achị' 386
'na-akpọ' 368
'na-enye' 351
'na-achọ' 341
'aka.' 317
'ga-abụ' 310
'kwuru,' 302
'sịrị:' 294
'anya,' 279
'na-arụ' 278
'ike.' 278
'mma.' 276
'niile.' 266
'na-eche' 261
'ọzọ,' 250
'anyị.' 246
'Baịbụl.' 243
'na-atụ' 237
'anyị,' 235
'bụ,' 234
'Naịjirịa.' 223
'm,' 221
'ọma.' 220
'ihe.' 220
'm.' 218
'ga-adị' 216
'na-abịa' 215
'na-eji' 210


In [ ]:
import unicodedata

def has_punctuation(token):
    return any(
        unicodedata.category(ch).startswith("P")
        for ch in token
    )

unicode_punct = [
    (word, freq)
    for word, freq in uncovered
    if has_punctuation(word)
]

print(
    "Uncovered types containing Unicode punctuation:",
    len(unicode_punct)
)

print(
    "Occurrences:",
    sum(freq for _, freq in unicode_punct)
)

Uncovered types containing Unicode punctuation: 22988
Occurrences: 140328


In [ ]:
with open(TRAINING_CORPUS_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Number of lines:", len(lines))

for line in lines[:20]:
    print(repr(line))

Number of lines: 35078
'Sowore Revolution: Ka ekwe si akụ maka ngagharịiwe e ji maka ya nwụchie Sowore BBC Igbo kwụ chịm iwetara gị ihe na-aga ka a na-ekwu okwu ngagharịiwe na mpaghara dị iche iche na Naịjirịa.\n'
'Ofala Festival: Ka mmemme Ofala sị aga n’ Onitsha taa Oge eruola ọzọ mgbe Obi na-achị Onitsha ga-apụta n’ ebube ya.\n'
'Onitsha tanker fire: Ndị ọkụ lara ihe ha n’ iyi na-ekwu etu ọ dị ha Obi nke Onitsha esonyela na ndị na-eti ndị ọkụ tanka a metụtara aka n’ obi.\n'
'Hilda Dokubọ kọwara ihe ise dị mkpa nne ọbụla kwesiri ịma Hilda Dokubọ bụ onye na-eme ihe nkiri na-enye ndị nne ezigbo ndụmọdụ.\n'
'Presidential Election: A napụrụ ọtụtụ n’ ọwụwa anyanwụ ohere itu vootu-Peter Obi Peter Obi atụọla arịrị na usoro a gbasoro mee ntuliaka n’ Ọwụwa anyanwụ dị iche na nke Ugwu-Awụsa.\n'
'Ọgba mbọ: Ihe mere m ji ewe ụmụ nwaanyị n’ ọrụ karịa ụmụnwoke Ụkamaka Okoye kọwara BBC Igbo etu o siri bido n’ iji akpakara igwe rụpụtawa ụzụ igwe nke ọgbatumtum ji arụ ọrụ.\n'
"Onye okachamara n’ ihe 